# Taller III — Limpieza, transformación y validación de ventas con PySpark en Databricks

**Curso:** Gestión de datos y tecnologías Big Data (MCDI502) · Semana 3 · Sumativa 4
**Grupo:** 1 · **Integrantes:** Yerko Gallardo, Sebastián Rojas, José Ignacio Rodríguez

**Caso:** consolidar las ventas del último trimestre de 2024 de las sucursales de Viña del Mar, Santiago y Concepción, depurar sus problemas de calidad y dejar un dataset limpio y validado para el análisis con Spark SQL.

| Parte del enunciado | Sección |
|---|---|
| 1. Carga e integración (ID3.1) | 1 |
| 2. Limpieza y depuración (ID4.2) | 2 |
| 3. Transformación y análisis con DataFrame (ID4.1) | 3 |
| 4. Validación y control de calidad (ID4.3) | 4 |
| 5. Análisis con Spark SQL (ID3.3) | 5 |

## 1. Carga e integración de datos (ID3.1)

### 1.1 Ruta de los archivos

**Propósito:** leer los tres CSV de Canvas desde el almacenamiento de Databricks.

**Decisión de almacenamiento:** Los tres CSV de Canvas se cargaron en el volumen de Unity Catalog /Volumes/workspace/data/data del workspace de Databricks Free Edition. Databricks recomienda los volúmenes de Unity Catalog para almacenar archivos y documenta que DBFS root y los montajes heredados están obsoletos. La ruta base se define en RUTA_BASE para facilitar la ejecución en otro workspace. https://docs.databricks.com/aws/en/dbfs/

In [0]:
from pyspark.sql import functions as F

RUTA_BASE = "/Volumes/workspace/data/data"   # Volume de Unity Catalog (Free Edition no permite la raíz de DBFS)
ARCHIVOS = {
    "Viña del Mar": "ventas_vina_del_mar.csv",
    "Santiago": "ventas_santiago.csv",
    "Concepción": "ventas_concepcion.csv",
}

### 1.2 Carga de cada sucursal en su propio DataFrame

**Regla de lectura:**

- `header=True`: la primera fila trae los nombres de las columnas.
- **Todas las columnas como texto** (sin `inferSchema`). Las fechas y los precios vienen en varios formatos y con símbolos, así que las conversiones se hacen de forma explícita en la Parte 2, donde se cuentan las que fallan.
- `escape='"'`: algunos nombres de producto traen comillas dobles escapadas, como `"Monitor 27"""`. Con el escape por defecto de Spark (`\`), el valor queda como `Monitor 27""`; con `escape='"'` se lee correctamente como `Monitor 27"`.
- Un campo vacío se lee como `NULL`, que es lo que permite perfilar los nulos de `categoria`.

In [0]:
def leer_csv(archivo):
    """Lee un CSV de ventas con todas las columnas como texto."""
    return (spark.read
            .option("header", True)
            .option("escape", '"')      # comillas dobles escapadas dentro de un campo: "Monitor 27"""
            .option("encoding", "UTF-8")
            .csv(f"{RUTA_BASE}/{archivo}"))


df_vina = leer_csv(ARCHIVOS["Viña del Mar"])
df_santiago = leer_csv(ARCHIVOS["Santiago"])
df_concepcion = leer_csv(ARCHIVOS["Concepción"])

### 1.3 Inspección inicial: esquema, primeras 5 filas y sucursal de origen

Para cada DataFrame se muestra `printSchema()` y `show(5)`, y se verifica que el archivo contenga **una sola sucursal** y que corresponda a la del nombre del archivo.

In [0]:
entradas = {"Viña del Mar": df_vina, "Santiago": df_santiago, "Concepción": df_concepcion}
columnas_referencia = df_vina.columns

for sucursal, df in entradas.items():
    print(f"\n===== {sucursal}: {df.count()} filas =====")
    df.printSchema()
    df.show(5, truncate=False)
    sucursales = [fila.sucursal for fila in df.select("sucursal").distinct().collect()]
    assert sucursales == [sucursal], f"{sucursal}: el archivo contiene {sucursales}"
    assert df.columns == columnas_referencia, f"{sucursal}: columnas distintas a las de Viña del Mar"
    print(f"Sucursal de origen verificada: {sucursales[0]} · mismas {len(df.columns)} columnas que el resto")


===== Viña del Mar: 105 filas =====
root
 |-- id_transaccion: string (nullable = true)
 |-- fecha: string (nullable = true)
 |-- id_producto: string (nullable = true)
 |-- producto_nombre: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- cantidad: string (nullable = true)
 |-- precio_unitario: string (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- sucursal: string (nullable = true)

+--------------+----------+-----------+-----------------+--------------+--------+---------------+----------+------------+
|id_transaccion|fecha     |id_producto|producto_nombre  |categoria     |cantidad|precio_unitario|id_cliente|sucursal    |
+--------------+----------+-----------+-----------------+--------------+--------+---------------+----------+------------+
|TRX10030      |01/10/2024|PROD889    |Mouse Inalámbrico|Almacenamiento|8       |79990          |CLI1158   |Viña del Mar|
|TRX10065      |2024/10/13|PROD403    |Teclado Mecánico |Almacenamiento|-1      |1    

### 1.4 Unión en `df_ventas_unificado` y verificación de integridad

**Regla:** los tres archivos tienen las mismas 9 columnas en el mismo orden, todas como texto. Se unen con `unionByName`, que empareja por nombre de columna y no por posición, y se verifica que el total sea exactamente la suma de las tres entradas. La unión se guarda en caché porque todas las etapas siguientes parten de ella.

In [0]:
df_ventas_unificado = (df_vina
                       .unionByName(df_santiago)
                       .unionByName(df_concepcion)
                       .cache())

conteos = {sucursal: df.count() for sucursal, df in entradas.items()}
suma_entradas = sum(conteos.values())
total_unificado = df_ventas_unificado.count()

for sucursal, n in conteos.items():
    print(f"{sucursal:13s}: {n:4d} filas")
print(f"{'Suma':13s}: {suma_entradas:4d} filas")
print(f"{'Unificado':13s}: {total_unificado:4d} filas")
assert total_unificado == suma_entradas, "La unión no conserva todas las filas"
print("\nIntegridad verificada: df_ventas_unificado tiene la suma exacta de las tres entradas.")

df_ventas_unificado.groupBy("sucursal").count().orderBy("sucursal").show()

Viña del Mar :  105 filas
Santiago     :  125 filas
Concepción   :   85 filas
Suma         :  315 filas
Unificado    :  315 filas

Integridad verificada: df_ventas_unificado tiene la suma exacta de las tres entradas.
+------------+-----+
|    sucursal|count|
+------------+-----+
|  Concepción|   85|
|    Santiago|  125|
|Viña del Mar|  105|
+------------+-----+



### 1.5 Perfil inicial de la unión cruda

Línea base para el informe de calidad de la Parte 4: filas, nulos por columna y duplicados completos. Se conserva `df_ventas_crudo` como referencia de la unión **antes** de cualquier limpieza.

In [0]:
df_ventas_crudo = df_ventas_unificado   # referencia de la unión cruda para la comparación antes/después


def perfil_calidad(df, nombre):
    """Filas, nulos por columna y duplicados completos (filas excedentes respecto de las distintas)."""
    filas = df.count()
    distintas = df.distinct().count()
    print(f"{nombre}: {filas} filas · {distintas} distintas · {filas - distintas} duplicados completos")
    df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show(truncate=False)


perfil_calidad(df_ventas_crudo, "df_ventas_unificado (crudo)")

df_ventas_unificado (crudo): 315 filas · 300 distintas · 15 duplicados completos
+--------------+-----+-----------+---------------+---------+--------+---------------+----------+--------+
|id_transaccion|fecha|id_producto|producto_nombre|categoria|cantidad|precio_unitario|id_cliente|sucursal|
+--------------+-----+-----------+---------------+---------+--------+---------------+----------+--------+
|0             |0    |0          |0              |46       |0       |0              |0         |0       |
+--------------+-----+-----------+---------------+---------+--------+---------------+----------+--------+



## 2. Limpieza y depuración de datos (ID4.2)

Las reglas se aplican **en orden** sobre `df_ventas_unificado`. Cada etapa registra en una bitácora sus filas de entrada, de salida y retiradas, con un solo motivo por fila, para que los conteos se puedan conciliar sin contar dos veces una misma fila.

In [0]:
bitacora = []          # (etapa, filas de entrada, filas de salida, filas retiradas o modificadas, regla)


def registrar(etapa, entrada, salida, regla, afectadas=None):
    afectadas = entrada - salida if afectadas is None else afectadas
    bitacora.append((etapa, entrada, salida, afectadas, regla))
    print(f"[{etapa}] entrada {entrada} · salida {salida} · afectadas {afectadas} · {regla}")

### 2.1 Gestión de nulos en `categoria`

**Perfil:** se cuentan los nulos y los textos vacíos o con solo espacios de `categoria` por sucursal. Luego se revisa si la categoría faltante se podría **recuperar** a partir del producto: si cada `id_producto` o cada nombre de producto tuviera siempre la misma categoría, se podría completar con ella.

In [0]:
categoria_faltante = F.col("categoria").isNull() | (F.trim(F.col("categoria")) == "")

print("Categoría faltante (nula o vacía) por sucursal:")
(df_ventas_unificado
 .groupBy("sucursal")
 .agg(F.count("*").alias("filas"),
      F.sum(F.col("categoria").isNull().cast("int")).alias("nulas"),
      F.sum((F.trim(F.col("categoria")) == "").cast("int")).alias("vacias"))
 .withColumn("pct_faltante", F.round((F.col("nulas") + F.col("vacias")) / F.col("filas") * 100, 1))
 .orderBy("sucursal")
 .show())

print("Valores de categoria:")
df_ventas_unificado.groupBy("categoria").count().orderBy(F.desc("count")).show()

# ¿Se puede recuperar la categoría desde el producto?
con_categoria = df_ventas_unificado.filter(~categoria_faltante)
print("Cantidad de categorías distintas por id_producto (entre las filas con categoría):")
(con_categoria.groupBy("id_producto").agg(F.countDistinct("categoria").alias("categorias"))
 .groupBy("categorias").count().orderBy("categorias").show())
print("Categorías en que aparece un mismo producto (nombre en minúsculas y sin tildes):")
(con_categoria.groupBy(F.translate(F.lower("producto_nombre"), "áéíóú", "aeiou").alias("producto"))
 .agg(F.countDistinct("categoria").alias("categorias"),
      F.sort_array(F.collect_set("categoria")).alias("valores"))
 .orderBy("producto").show(truncate=False))

Categoría faltante (nula o vacía) por sucursal:
+------------+-----+-----+------+------------+
|    sucursal|filas|nulas|vacias|pct_faltante|
+------------+-----+-----+------+------------+
|  Concepción|   85|   11|     0|        12.9|
|    Santiago|  125|   19|     0|        15.2|
|Viña del Mar|  105|   16|     0|        15.2|
+------------+-----+-----+------+------------+

Valores de categoria:
+--------------+-----+
|     categoria|count|
+--------------+-----+
|   Periféricos|   55|
|   Computación|   55|
|         Audio|   55|
|Almacenamiento|   53|
|         Redes|   51|
|          NULL|   46|
+--------------+-----+

Cantidad de categorías distintas por id_producto (entre las filas con categoría):
+----------+-----+
|categorias|count|
+----------+-----+
|         1|  203|
|         2|   21|
|         3|    1|
+----------+-----+

Categorías en que aparece un mismo producto (nombre en minúsculas y sin tildes):
+---------------------+----------+--------------------------------------

**Estrategia: imputar `"Sin categoría"` con `.fillna()`. No se eliminan las filas.**

- **No se puede recuperar** la categoría desde el producto: el mismo `id_producto` aparece en distintas categorías, y un mismo producto (por ejemplo, una webcam) figura a la vez en Audio, Almacenamiento, Redes y otras. No existe una relación producto → categoría que permita completar el dato.
- **Eliminar (`.dropna()`) descartaría cerca del 15 % de las ventas**, que tienen fecha, producto, cantidad y precio válidos. Eso sesgaría los totales por sucursal de la Parte 5.
- `"Sin categoría"` deja explícito que el dato falta, sin inventar una categoría. Los vacíos se unifican primero como nulos (`trim` y vacío → `NULL`) para que `.fillna()` los cubra todos.
- **Límite:** las ventas sin categoría quedan agrupadas en una categoría propia en la consulta por categoría (Parte 5).

In [0]:
entrada = df_ventas_unificado.count()
faltantes = df_ventas_unificado.filter(categoria_faltante).count()

df_categoria = (df_ventas_unificado
                .withColumn("categoria", F.when(categoria_faltante, None).otherwise(F.trim("categoria")))
                .fillna({"categoria": "Sin categoría"}))

assert df_categoria.filter(F.col("categoria").isNull()).count() == 0
assert df_categoria.filter(F.col("categoria") == "Sin categoría").count() == faltantes
registrar("2.1 categoria", entrada, df_categoria.count(), "nulos y vacíos imputados con 'Sin categoría'", faltantes)
df_categoria.groupBy("categoria").count().orderBy(F.desc("count")).show()

[2.1 categoria] entrada 315 · salida 315 · afectadas 46 · nulos y vacíos imputados con 'Sin categoría'
+--------------+-----+
|     categoria|count|
+--------------+-----+
|   Periféricos|   55|
|   Computación|   55|
|         Audio|   55|
|Almacenamiento|   53|
|         Redes|   51|
| Sin categoría|   46|
+--------------+-----+



### 2.2 Eliminación de duplicados completos

**Regla:** un duplicado completo es una fila idéntica a otra en **todas** las columnas originales. Se cuentan como filas **excedentes** (total − distintas) y se eliminan con `.dropDuplicates()`, que conserva una copia de cada fila. No hay columnas técnicas agregadas que hagan única cada fila, así que la comparación usa las 9 columnas del archivo. Además se revisa que, tras eliminarlos, cada `id_transaccion` aparezca una sola vez.

In [0]:
entrada = df_categoria.count()
distintas = df_categoria.distinct().count()
print(f"Filas: {entrada} · distintas: {distintas} · duplicados completos (excedentes): {entrada - distintas}")

print("Grupos de filas idénticas. La columna 'veces' indica cuántas veces aparece cada fila.")
duplicadas = (df_categoria.groupBy(*df_categoria.columns).count()
              .filter(F.col("count") > 1).withColumnRenamed("count", "veces"))
duplicadas.select("id_transaccion", "sucursal", "fecha", "producto_nombre", "cantidad", "precio_unitario", "veces") \
          .orderBy("sucursal", "id_transaccion").show(20, truncate=False)

df_sin_duplicados = df_categoria.dropDuplicates()

ids_repetidos = df_sin_duplicados.groupBy("id_transaccion").count().filter(F.col("count") > 1).count()
assert df_sin_duplicados.count() == distintas
assert ids_repetidos == 0, "Hay id_transaccion repetidos con contenido distinto"
registrar("2.2 duplicados", entrada, df_sin_duplicados.count(), "dropDuplicates() sobre las 9 columnas")
print("Cada id_transaccion queda una sola vez:", ids_repetidos == 0)

Filas: 315 · distintas: 300 · duplicados completos (excedentes): 15
Filas repetidas (cada una aparece 'veces' veces):
+--------------+------------+----------+---------------------+--------+---------------+-----+
|id_transaccion|sucursal    |fecha     |producto_nombre      |cantidad|precio_unitario|veces|
+--------------+------------+----------+---------------------+--------+---------------+-----+
|TRX30000      |Concepción  |2024-10-03|Router WIFI          |9       |89990          |2    |
|TRX30018      |Concepción  |18/10/2024|teclado mecanico     |2       |$ 159990       |2    |
|TRX30022      |Concepción  |2024/12/26|Laptop Gamer         |2       |45990          |2    |
|TRX30030      |Concepción  |2024/11/23|Monitor 27 Pulgadas  |3       |$ 79990        |2    |
|TRX30031      |Concepción  |24/10/2024|auriculares bluetooth|8       |$ 89990        |2    |
|TRX20004      |Santiago    |2024/10/09|Mouse Inalambrico    |3       |$50            |2    |
|TRX20026      |Santiago    |2024/10

### 2.3 Corrección de inconsistencias

#### a) `producto_nombre`: formato consistente

**Perfil:** el mismo producto aparece escrito de muchas formas: mayúsculas (`IMPRESORA LASER`), tildes (`Láser`), sinónimos (`Laptop Gaming` / `Laptop Gamer`, `SSD 1TB` / `Disco SSD 1TB`, `RAM 16GB` / `Memoria RAM 16GB`) y abreviaturas (`Monitor 27"`).

**Regla, en este orden:**

1. `lower` y `trim`: minúsculas, sin espacios en los extremos.
2. `translate`: se quitan las tildes (`láser` → `laser`).
3. `regexp_replace`: el símbolo `"` de pulgadas pasa a la palabra `pulgadas`, y los espacios repetidos se reducen a uno.
4. Un **diccionario de sinónimos** lleva cada variante a un nombre canónico.

El resultado queda en minúsculas, como sugiere el enunciado.

In [0]:
SINONIMOS = {   # variante normalizada -> nombre canónico
    "laptop gaming": "laptop gamer",
    "ssd 1tb": "disco ssd 1tb",
    "ram 16gb": "memoria ram 16gb",
}
mapa_sinonimos = F.create_map(*[F.lit(x) for par in SINONIMOS.items() for x in par])


def normalizar_producto(columna):
    base = F.lower(F.trim(columna))
    base = F.translate(base, "áéíóúüñ", "aeiouun")
    base = F.regexp_replace(base, '"', " pulgadas")
    base = F.trim(F.regexp_replace(base, r"\s+", " "))
    return F.coalesce(F.element_at(mapa_sinonimos, base), base)


df_productos = (df_sin_duplicados
                .withColumn("producto_original", F.col("producto_nombre"))
                .withColumn("producto_nombre", normalizar_producto(F.col("producto_nombre"))))

variantes_antes = df_productos.select("producto_original").distinct().count()
variantes_despues = df_productos.select("producto_nombre").distinct().count()
print(f"Variantes de producto_nombre: {variantes_antes} antes → {variantes_despues} después")

print("Ejemplos antes → después (todas las variantes):")
(df_productos.groupBy("producto_nombre")
 .agg(F.sort_array(F.collect_set("producto_original")).alias("variantes_originales"), F.count("*").alias("filas"))
 .orderBy("producto_nombre").show(truncate=False))

Variantes de producto_nombre: 31 antes → 10 después
Ejemplos antes → después (todas las variantes):
+---------------------+---------------------------------------------------------------------+-----+
|producto_nombre      |variantes_originales                                                 |filas|
+---------------------+---------------------------------------------------------------------+-----+
|auriculares bluetooth|[AURICULARES BLUETOOTH, Auriculares Bluetooth, auriculares bluetooth]|28   |
|disco ssd 1tb        |[Disco SSD 1TB, SSD 1TB, disco ssd 1tb]                              |32   |
|impresora laser      |[IMPRESORA LASER, Impresora Láser, impresora laser]                  |35   |
|laptop gamer         |[LAPTOP GAMER, Laptop Gamer, Laptop Gaming, laptop gamer]            |38   |
|memoria ram 16gb     |[Memoria RAM 16GB, RAM 16GB, memoria ram 16gb]                       |22   |
|monitor 27 pulgadas  |[Monitor 27 Pulgadas, Monitor 27", monitor 27 pulgadas]              |28   |


#### b) `fecha`: tipo `Date` estándar

**Perfil:** cada sucursal mezcla cuatro formatos: `aaaa-mm-dd`, `aaaa/mm/dd`, `dd/mm/aaaa` y `dd-mm-aaaa`.

**Regla:** primero se unifica el separador (`/` → `-`). Luego se elige el patrón según la posición del año: si el texto empieza con 4 dígitos se lee como `yyyy-MM-dd`, y si empieza con 2 dígitos, como `dd-MM-yyyy`. La conversión usa `try_to_timestamp`, que devuelve `NULL` en vez de un error si una fecha no es válida, y se cuentan esos casos.

**Por qué día/mes y no mes/día:** en los formatos con el año al final, muchas fechas tienen el primer número mayor que 12 (por ejemplo `29/11/2024`), lo que solo es posible si ese número es el día. Además, leídas como mes/día, la mayoría quedaría fuera del último trimestre. La celda comprueba ambas cosas.

In [0]:
fecha_texto = F.regexp_replace(F.trim(F.col("fecha")), "/", "-")
fecha_convertida = F.to_date(
    F.when(fecha_texto.rlike(r"^\d{4}-\d{2}-\d{2}$"), F.try_to_timestamp(fecha_texto, F.lit("yyyy-MM-dd")))
     .when(fecha_texto.rlike(r"^\d{2}-\d{2}-\d{4}$"), F.try_to_timestamp(fecha_texto, F.lit("dd-MM-yyyy"))))

df_fechas = (df_productos
             .withColumn("fecha_original", F.col("fecha"))
             .withColumn("fecha", fecha_convertida))

# Evidencia de la interpretación día/mes en los formatos con el año al final
año_al_final = df_fechas.filter(F.col("fecha_original").rlike(r"^\d{2}[-/]\d{2}[-/]\d{4}$"))
primer_numero = F.substring("fecha_original", 1, 2).cast("int")
segundo_numero = F.substring("fecha_original", 4, 2).cast("int")
print("Fechas con el año al final:", año_al_final.count(),
      "· primer número > 12 (solo puede ser día):", año_al_final.filter(primer_numero > 12).count(),
      "· leídas como mes/día quedarían fuera de oct–dic:",
      año_al_final.filter(~primer_numero.isin(10, 11, 12)).count())
assert año_al_final.filter(segundo_numero > 12).count() == 0, "Hay un mes mayor que 12 en formato día/mes"

print("\nFormato detectado → ejemplos de conversión:")
(df_fechas.withColumn("formato", F.regexp_replace("fecha_original", r"\d", "9"))
 .groupBy("formato").agg(F.count("*").alias("filas"), F.first("fecha_original").alias("ejemplo_original"),
                         F.first("fecha").alias("ejemplo_convertido"))
 .orderBy("formato").show(truncate=False))

fechas_ausentes = df_fechas.filter(F.col("fecha_original").isNull()).count()
fechas_fallidas = df_fechas.filter(F.col("fecha_original").isNotNull() & F.col("fecha").isNull()).count()
print(f"Fechas ausentes en el original: {fechas_ausentes} · conversiones fallidas: {fechas_fallidas}")
df_fechas.agg(F.min("fecha").alias("fecha_minima"), F.max("fecha").alias("fecha_maxima"),
              F.countDistinct(F.month("fecha")).alias("meses_distintos")).show()

Fechas con el año al final: 136 · primer número > 12 (solo puede ser día): 83 · leídas como mes/día quedarían fuera de oct–dic: 121

Formato detectado → ejemplos de conversión:
+----------+-----+----------------+------------------+
|formato   |filas|ejemplo_original|ejemplo_convertido|
+----------+-----+----------------+------------------+
|99-99-9999|68   |05-12-2024      |2024-12-05        |
|99/99/9999|68   |23/12/2024      |2024-12-23        |
|9999-99-99|84   |2024-11-14      |2024-11-14        |
|9999/99/99|80   |2024/10/23      |2024-10-23        |
+----------+-----+----------------+------------------+

Fechas ausentes en el original: 0 · conversiones fallidas: 0
+------------+------------+---------------+
|fecha_minima|fecha_maxima|meses_distintos|
+------------+------------+---------------+
|  2024-10-01|  2024-12-29|              3|
+------------+------------+---------------+



#### c) `precio_unitario`: tipo numérico

**Perfil:** los precios vienen como texto con prefijo `$` o `$ ` (por ejemplo `$ 199990`, `$45990` o `79990`). Son pesos chilenos enteros, sin separador de miles ni decimales.

**Regla:** `regexp_replace` elimina el símbolo `$` y los espacios, y `try_cast` convierte el resultado a `DECIMAL(12,0)`. Se usa un tipo decimal (exacto) y no `double`, porque son importes. Si un valor no se puede convertir, `try_cast` devuelve `NULL` y se cuenta. Del mismo modo se convierte `cantidad` a entero.

In [0]:
df_tipado = (df_fechas
             .withColumn("precio_original", F.col("precio_unitario"))
             .withColumn("precio_unitario",
                         F.expr("try_cast(regexp_replace(precio_unitario, '[$\\\\s]', '') AS DECIMAL(12,0))"))
             .withColumn("cantidad_original", F.col("cantidad"))
             .withColumn("cantidad", F.expr("try_cast(trim(cantidad) AS INT)")))

print("Formatos de precio → ejemplos de conversión:")
(df_tipado.withColumn("formato", F.regexp_replace("precio_original", r"\d", "9"))
 .groupBy("formato").agg(F.count("*").alias("filas"), F.first("precio_original").alias("ejemplo_original"),
                         F.first("precio_unitario").alias("ejemplo_convertido"))
 .orderBy(F.desc("filas")).show(truncate=False))

for columna, original in [("precio_unitario", "precio_original"), ("cantidad", "cantidad_original")]:
    ausentes = df_tipado.filter(F.col(original).isNull()).count()
    fallidas = df_tipado.filter(F.col(original).isNotNull() & F.col(columna).isNull()).count()
    print(f"{columna}: ausentes en el original {ausentes} · conversiones fallidas {fallidas}")

Formatos de precio → ejemplos de conversión:
+---------+-----+----------------+------------------+
|formato  |filas|ejemplo_original|ejemplo_convertido|
+---------+-----+----------------+------------------+
|999999   |66   |299990          |299990            |
|$ 999999 |59   |$ 129990        |129990            |
|$999999  |50   |$299990         |299990            |
|$ 99999  |47   |$ 45990         |45990             |
|99999    |37   |89990           |89990             |
|$99999   |26   |$45990          |45990             |
|$9999999 |4    |$8888888        |8888888           |
|9999999  |2    |9999990         |9999990           |
|9        |2    |1               |1                 |
|$ 9      |2    |$ 1             |1                 |
|$ 9999999|1    |$ 9999990       |9999990           |
|$99      |1    |$50             |50                |
|99       |1    |50              |50                |
|$9       |1    |$1              |1                 |
|$ 99     |1    |$ 50            |50 

#### d) Filas que no se pudieron convertir y resultado de la limpieza

Las filas cuya fecha, precio o cantidad no se haya podido convertir se separan en `df_descartes` con su motivo, en vez de dejar nulos que alteren los cálculos. Cada fila recibe **un solo motivo**: el primero que no cumple, en el orden fecha → precio → cantidad. Después se revisa que la normalización no haya creado duplicados nuevos.

In [0]:
COLUMNAS_TRAZA = ["id_transaccion", "sucursal", "fecha_original", "producto_original", "cantidad_original",
                  "precio_original"]

motivo_conversion = (F.when(F.col("fecha").isNull(), "fecha no convertible")
                      .when(F.col("precio_unitario").isNull(), "precio no convertible")
                      .when(F.col("cantidad").isNull(), "cantidad no convertible"))

entrada = df_tipado.count()
df_descartes = (df_tipado.withColumn("motivo", motivo_conversion).filter(F.col("motivo").isNotNull())
                .select(*COLUMNAS_TRAZA, "motivo", F.lit("2.3 conversión").alias("etapa")))
df_limpio_tipos = df_tipado.filter(motivo_conversion.isNull())
registrar("2.3 conversión", entrada, df_limpio_tipos.count(),
          "producto normalizado; fecha, precio y cantidad tipados; no convertibles separados")

# La normalización no debe producir duplicados nuevos en las columnas originales ya corregidas
columnas_negocio = ["id_transaccion", "fecha", "id_producto", "producto_nombre", "categoria",
                    "cantidad", "precio_unitario", "id_cliente", "sucursal"]
nuevos = df_limpio_tipos.count() - df_limpio_tipos.select(*columnas_negocio).distinct().count()
print("Duplicados nuevos creados por la normalización:", nuevos)
assert nuevos == 0

print("Esquema después de la limpieza:")
df_limpio_tipos.select(*columnas_negocio).printSchema()
df_limpio_tipos.select(*columnas_negocio).show(5, truncate=False)

[2.3 conversión] entrada 300 · salida 300 · afectadas 0 · producto normalizado; fecha, precio y cantidad tipados; no convertibles separados
Duplicados nuevos creados por la normalización: 0
Esquema después de la limpieza:
root
 |-- id_transaccion: string (nullable = true)
 |-- fecha: date (nullable = true)
 |-- id_producto: string (nullable = true)
 |-- producto_nombre: string (nullable = true)
 |-- categoria: string (nullable = false)
 |-- cantidad: integer (nullable = true)
 |-- precio_unitario: decimal(12,0) (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- sucursal: string (nullable = true)

+--------------+----------+-----------+---------------------+-------------+--------+---------------+----------+--------+
|id_transaccion|fecha     |id_producto|producto_nombre      |categoria    |cantidad|precio_unitario|id_cliente|sucursal|
+--------------+----------+-----------+---------------------+-------------+--------+---------------+----------+--------+
|TRX20019      |202

## 3. Transformación y análisis con DataFrame (ID4.1)

### 3.1 Filtrado de cantidades inválidas

**Regla:** una venta con cantidad **menor o igual a cero** no representa una venta efectiva (las cantidades negativas parecen devoluciones o errores de digitación, y el cero no mueve unidades). Se eliminan con `.filter()` y se registran en `df_descartes` con su motivo.

In [0]:
entrada = df_limpio_tipos.count()
print("Cantidades no positivas por valor:")
df_limpio_tipos.filter(F.col("cantidad") <= 0).groupBy("cantidad").count().orderBy("cantidad").show()

df_cantidad_invalida = df_limpio_tipos.filter(F.col("cantidad") <= 0)
df_descartes = df_descartes.unionByName(
    df_cantidad_invalida.select(*COLUMNAS_TRAZA, F.lit("cantidad <= 0").alias("motivo"),
                                F.lit("3.1 filtro").alias("etapa")))

df_cantidad_valida = df_limpio_tipos.filter(F.col("cantidad") > 0)
registrar("3.1 cantidad", entrada, df_cantidad_valida.count(), "filter(cantidad > 0)")
assert df_cantidad_valida.filter(F.col("cantidad") <= 0).count() == 0

Cantidades no positivas por valor:
+--------+-----+
|cantidad|count|
+--------+-----+
|      -5|   10|
|      -2|    8|
|      -1|    4|
|       0|    4|
+--------+-----+

[3.1 cantidad] entrada 300 · salida 274 · afectadas 26 · filter(cantidad > 0)


### 3.2 Columnas calculadas: `precio_total` y `dia_semana`

- `precio_total = cantidad × precio_unitario`, en `DECIMAL(14,0)`, el mismo tipo exacto que el precio.
- `dia_semana` se calcula desde la fecha estandarizada con la convención **lunes = 1 … domingo = 7** (ISO 8601). `dayofweek()` de Spark numera desde el domingo (domingo = 1), así que se convierte con `(dayofweek + 5) % 7 + 1`. Se agrega `dia_semana_nombre` para facilitar la lectura.

In [0]:
NOMBRES_DIA = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]
dia_iso = ((F.dayofweek("fecha") + 5) % 7 + 1)

df_ventas_preprocesado = (df_cantidad_valida
                          .withColumn("precio_total", (F.col("cantidad") * F.col("precio_unitario")).cast("decimal(14,0)"))
                          .withColumn("dia_semana", dia_iso)
                          .withColumn("dia_semana_nombre", F.element_at(F.array(*[F.lit(d) for d in NOMBRES_DIA]), dia_iso))
                          .select(*columnas_negocio, "precio_total", "dia_semana", "dia_semana_nombre")
                          .cache())

# Comprobación de la convención con fechas conocidas
prueba = spark.createDataFrame([("2024-12-30",), ("2024-12-29",)], "f STRING").select(
    F.to_date("f").alias("fecha"), ((F.dayofweek(F.to_date("f")) + 5) % 7 + 1).alias("dia_semana"))
assert [r.dia_semana for r in prueba.collect()] == [1, 7], "30/12/2024 es lunes y 29/12/2024 es domingo"
print("Convención verificada: 30/12/2024 (lunes) = 1 · 29/12/2024 (domingo) = 7")

df_ventas_preprocesado.select("id_transaccion", "fecha", "dia_semana", "dia_semana_nombre", "cantidad",
                              "precio_unitario", "precio_total").show(5, truncate=False)
print("Ventas por día de la semana:")
df_ventas_preprocesado.groupBy("dia_semana", "dia_semana_nombre").count().orderBy("dia_semana").show()

Convención verificada: 30/12/2024 (lunes) = 1 · 29/12/2024 (domingo) = 7
+--------------+----------+----------+-----------------+--------+---------------+------------+
|id_transaccion|fecha     |dia_semana|dia_semana_nombre|cantidad|precio_unitario|precio_total|
+--------------+----------+----------+-----------------+--------+---------------+------------+
|TRX20019      |2024-11-14|4         |jueves           |5       |299990         |1499950     |
|TRX20010      |2024-11-25|1         |lunes            |2       |349990         |699980      |
|TRX20000      |2024-11-08|5         |viernes          |8       |349990         |2799920     |
|TRX20026      |2024-10-23|3         |miércoles        |8       |159990         |1279920     |
|TRX20094      |2024-12-23|1         |lunes            |5       |89990          |449950      |
+--------------+----------+----------+-----------------+--------+---------------+------------+
only showing top 5 rows
Ventas por día de la semana:
+----------+-------

### 3.3 Análisis con `distinct()`: sucursales y categorías únicas

In [0]:
sucursales_unicas = df_ventas_preprocesado.select("sucursal").distinct()
categorias_unicas = df_ventas_preprocesado.select("categoria").distinct()

print(f"Sucursales únicas: {sucursales_unicas.count()}")
sucursales_unicas.orderBy("sucursal").show(truncate=False)
print(f"Categorías únicas: {categorias_unicas.count()} (incluye 'Sin categoría')")
categorias_unicas.orderBy("categoria").show(truncate=False)

Sucursales únicas: 3
+------------+
|sucursal    |
+------------+
|Concepción  |
|Santiago    |
|Viña del Mar|
+------------+

Categorías únicas: 6 (incluye 'Sin categoría')
+--------------+
|categoria     |
+--------------+
|Almacenamiento|
|Audio         |
|Computación   |
|Periféricos   |
|Redes         |
|Sin categoría |
+--------------+



### 3.4 Bitácora de limpieza y traspaso a la Parte 4

La bitácora concilia las filas etapa por etapa, y `df_descartes` guarda cada fila retirada con su motivo. Los duplicados no aparecen en `df_descartes`: quedan representados por la copia que se conserva.

In [0]:
registrar("Resultado", df_ventas_unificado.count(), df_ventas_preprocesado.count(), "df_ventas_preprocesado")
spark.createDataFrame(bitacora, "etapa STRING, filas_entrada BIGINT, filas_salida BIGINT, "
                                "filas_afectadas BIGINT, regla STRING").show(truncate=False)

print("Filas descartadas por motivo:")
df_descartes.groupBy("etapa", "motivo").count().orderBy("etapa").show(truncate=False)

# Conciliación: unificado − duplicados − descartes = preprocesado
total = df_ventas_unificado.count()
duplicados = total - df_ventas_unificado.distinct().count()
descartadas = df_descartes.count()
final = df_ventas_preprocesado.count()
print(f"{total} unificadas − {duplicados} duplicados − {descartadas} descartadas = {total - duplicados - descartadas}"
      f" · df_ventas_preprocesado: {final}")
assert total - duplicados - descartadas == final

print("\nEsquema de df_ventas_preprocesado:")
df_ventas_preprocesado.printSchema()
df_ventas_preprocesado.groupBy("sucursal").count().orderBy("sucursal").show()

[Resultado] entrada 315 · salida 274 · afectadas 41 · df_ventas_preprocesado
+--------------+-------------+------------+---------------+---------------------------------------------------------------------------------+
|etapa         |filas_entrada|filas_salida|filas_afectadas|regla                                                                            |
+--------------+-------------+------------+---------------+---------------------------------------------------------------------------------+
|2.1 categoria |315          |315         |46             |nulos y vacíos imputados con 'Sin categoría'                                     |
|2.2 duplicados|315          |300         |15             |dropDuplicates() sobre las 9 columnas                                            |
|2.3 conversión|300          |300         |0              |producto normalizado; fecha, precio y cantidad tipados; no convertibles separados|
|3.1 cantidad  |300          |274         |26             |filter(canti

## 4. Validación y control de calidad (ID4.3)
**Objetivo:** validar importes, detectar valores atípicos de precio unitario y comparar calidad antes/después.
**Criterio:** excluir importes nulos o negativos para definir ventas válidas; conservar atípicos plausibles marcados, sin eliminarlos automáticamente.
 Se utilizan los DataFrames creados en las secciones anteriores: `df_ventas_crudo`, `df_ventas_preprocesado` y `df_descartes`.


In [0]:
from pyspark.sql import functions as F

# 4.1 Control de importes: identificamos cualquier total nulo o negativo.
pre = df_ventas_preprocesado.count()
problemas_total = df_ventas_preprocesado.filter(
    F.col("precio_total").isNull() | (F.col("precio_total") < 0)
)
n_problemas_total = problemas_total.count()
print(f"Preprocesado: {pre} filas | Importes nulos o negativos: {n_problemas_total}")
problemas_total.select("id_transaccion", "sucursal", "cantidad", "precio_unitario", "precio_total").show(20, False)

# Solo si hay registros inválidos se separan; no se ocultan ni se imputan arbitrariamente.
# Se respeta una sola razón por registro, primero nulo y después negativo.
df_importes_validos = df_ventas_preprocesado.filter(
    F.col("precio_total").isNotNull() & (F.col("precio_total") >= 0)
)
assert pre == df_importes_validos.count() + n_problemas_total


Preprocesado: 274 filas | Importes nulos o negativos: 0
+--------------+--------+--------+---------------+------------+
|id_transaccion|sucursal|cantidad|precio_unitario|precio_total|
+--------------+--------+--------+---------------+------------+
+--------------+--------+--------+---------------+------------+



### 4.2 Detección de precios atípicos con rango intercuartílico (IQR)
Se calcula Q1 y Q3 sobre precios unitarios no nulos y no negativos. Un valor queda marcado si está fuera de
**[Q1 − 1,5 × IQR, Q3 + 1,5 × IQR]**. Los registros se mantienen marcados: no hay evidencia suficiente para concluir que un precio extremo sea incorrecto.
 `approxQuantile` es aproximado; el error relativo solicitado es 0,001.

In [0]:
precios_validos = df_importes_validos.filter(
    F.col("precio_unitario").isNotNull() & (F.col("precio_unitario") >= 0)
)
n_precios = precios_validos.count()
if n_precios:
    q1, q3 = precios_validos.approxQuantile("precio_unitario", [0.25, 0.75], 0.001)
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
else:
    q1 = q3 = iqr = limite_inferior = limite_superior = None

if n_precios == 0:
    print("No existen precios válidos: no es posible determinar los umbrales IQR.")
elif iqr == 0:
    print(f"Q1 = Q3 = {q1}. IQR = 0: se marcan los precios diferentes a {q1}; interpretar con cautela.")
else:
    print(f"Q1={q1:,.2f}; Q3={q3:,.2f}; IQR={iqr:,.2f}")
    print(f"Límite inferior={limite_inferior:,.2f}; límite superior={limite_superior:,.2f}")

if n_precios == 0:
    condicion_outlier = F.lit(False)
elif iqr == 0:
    condicion_outlier = F.col("precio_unitario") != F.lit(q1)
else:
    condicion_outlier = (
        (F.col("precio_unitario") < F.lit(limite_inferior)) |
        (F.col("precio_unitario") > F.lit(limite_superior))
    )

df_ventas_limpio = df_importes_validos.withColumn("es_outlier_precio", F.coalesce(condicion_outlier, F.lit(False))).cache()
total_final = df_ventas_limpio.count()
n_outliers = df_ventas_limpio.filter("es_outlier_precio").count()
print(f"Atípicos marcados: {n_outliers} de {total_final} ({100*n_outliers/total_final if total_final else 0:.2f} %) ")
df_ventas_limpio.filter("es_outlier_precio").select(
    "id_transaccion", "sucursal", "categoria", "producto_nombre", "precio_unitario", "cantidad", "precio_total"
).orderBy(F.desc("precio_unitario")).show(30, truncate=False)

Q1=79,990.00; Q3=299,990.00; IQR=220,000.00
Límite inferior=-250,010.00; límite superior=629,990.00
Atípicos marcados: 7 de 274 (2.55 %) 
+--------------+------------+-------------+---------------------+---------------+--------+------------+
|id_transaccion|sucursal    |categoria    |producto_nombre      |precio_unitario|cantidad|precio_total|
+--------------+------------+-------------+---------------------+---------------+--------+------------+
|TRX30009      |Concepción  |Redes        |impresora laser      |9999990        |2       |19999980    |
|TRX30045      |Concepción  |Sin categoría|laptop gamer         |9999990        |8       |79999920    |
|TRX20119      |Santiago    |Sin categoría|auriculares bluetooth|9999990        |7       |69999930    |
|TRX20001      |Santiago    |Sin categoría|disco ssd 1tb        |9999990        |7       |69999930    |
|TRX20037      |Santiago    |Audio        |mouse inalambrico    |9999990        |3       |29999970    |
|TRX20047      |Santiago    |A

### 4.3 Informe de calidad antes y después
Se comparan **solo las 9 columnas originales** para que las nuevas columnas calculadas y técnicas no alteren los conteos de nulos o duplicados. Las filas retiradas se concilian por etapa y se distinguen las filas modificadas de las descartadas.


In [0]:
columnas_originales = ["id_transaccion", "fecha", "id_producto", "producto_nombre",
                      "categoria", "cantidad", "precio_unitario", "id_cliente", "sucursal"]

def resumen_calidad(df, etapa):
    original = df.select(*columnas_originales)
    n = original.count()
    n_distintas = original.distinct().count()
    nulos = original.select(*[
        F.sum(F.col(c).isNull().cast("long")).alias(c) for c in columnas_originales
    ]).first().asDict()
    return {"etapa": etapa, "filas": n, "duplicados_completos": n - n_distintas}, nulos

resumen_antes, nulos_antes = resumen_calidad(df_ventas_crudo, "Antes (crudo)")
resumen_despues, nulos_despues = resumen_calidad(df_ventas_limpio, "Después (limpio)")
display(spark.createDataFrame([resumen_antes, resumen_despues]))
display(spark.createDataFrame([
    (c, int(nulos_antes[c] or 0), int(nulos_despues[c] or 0))
    for c in columnas_originales
], "columna STRING, nulos_antes BIGINT, nulos_despues BIGINT"))

# Duplicados sobre columnas originales: tras normalizar pueden coincidir filas que antes no eran idénticas.
print("Conciliación de filas:")
print(f"Crudas: {resumen_antes['filas']} | Preprocesadas: {pre} | Inválidas por total: {n_problemas_total} | Finales: {total_final}")
assert pre - n_problemas_total == total_final
assert resumen_antes["duplicados_completos"] >= 0
assert resumen_despues["duplicados_completos"] == 0, "Revisar duplicados posteriores a la normalización"
assert df_ventas_limpio.filter(F.col("precio_total").isNull() | (F.col("precio_total") < 0)).count() == 0
assert df_ventas_limpio.filter(F.col("cantidad") <= 0).count() == 0
print("Validaciones de cantidad e importe: OK")

# Evita interpretar conteos de nulos como equivalentes a vacíos en texto.
print("Nota: la columna categoria fue imputada con 'Sin categoría'; revisar ese grupo al interpretar sus agregados.")


duplicados_completos,etapa,filas
15,Antes (crudo),315
0,Después (limpio),274


columna,nulos_antes,nulos_despues
id_transaccion,0,0
fecha,0,0
id_producto,0,0
producto_nombre,0,0
categoria,46,0
cantidad,0,0
precio_unitario,0,0
id_cliente,0,0
sucursal,0,0


Conciliación de filas:
Crudas: 315 | Preprocesadas: 274 | Inválidas por total: 0 | Finales: 274
Validaciones de cantidad e importe: OK
Nota: la columna categoria fue imputada con 'Sin categoría'; revisar ese grupo al interpretar sus agregados.


## 5. Análisis con Spark SQL (ID3.3)
Las consultas trabajan sobre **el mismo DataFrame validado**, publicado como vista temporal `ventas_limpias`.
Se comprueba la unicidad de transacciones por sucursal antes de interpretar el conteo de transacciones.


In [0]:
ids_faltantes = df_ventas_limpio.filter(F.col("id_transaccion").isNull() | (F.trim("id_transaccion") == "")).count()
ids_repetidos_sucursal = df_ventas_limpio.groupBy("sucursal", "id_transaccion").count().filter("count > 1").count()
print(f"IDs ausentes: {ids_faltantes}; IDs repetidos dentro de una sucursal: {ids_repetidos_sucursal}")
if ids_faltantes or ids_repetidos_sucursal:
    print("Atención: COUNT(DISTINCT id_transaccion) podría diferir de COUNT(*). Revisar filas por transacción.")

# Publicar el DataFrame final para las tres consultas SQL.
df_ventas_limpio.createOrReplaceTempView("ventas_limpias")
print("Vista temporal ventas_limpias creada correctamente.")


IDs ausentes: 0; IDs repetidos dentro de una sucursal: 0
Vista temporal ventas_limpias creada correctamente.


### 5.1 Consulta SQL 1: top 10 de ventas de mayor importe
Se ordena por importe descendente y se utilizan columnas adicionales para desempatar.


In [0]:
%sql
SELECT id_transaccion, fecha, sucursal, producto_nombre, precio_total
FROM ventas_limpias
ORDER BY precio_total DESC, sucursal ASC, id_transaccion ASC
LIMIT 10

id_transaccion,fecha,sucursal,producto_nombre,precio_total
TRX20047,2024-12-07,Santiago,router wifi,79999992
TRX30045,2024-10-01,Concepción,laptop gamer,79999920
TRX20001,2024-11-23,Santiago,disco ssd 1tb,69999930
TRX20119,2024-12-19,Santiago,auriculares bluetooth,69999930
TRX10037,2024-11-10,Viña del Mar,webcam hd,35555552
TRX20037,2024-12-16,Santiago,mouse inalambrico,29999970
TRX30009,2024-10-20,Concepción,impresora laser,19999980
TRX20014,2024-10-15,Santiago,teclado mecanico,4499900
TRX10074,2024-11-25,Viña del Mar,teclado mecanico,4499900
TRX30019,2024-12-08,Concepción,laptop gamer,4049910


### Interpretación del Top 10 de ventas

La mayor transacción corresponde a Santiago, con un importe de **$79.999.992**, asociada a la venta de un router WiFi. Le sigue Concepción, con $79.999.920 por una laptop gamer.

De las diez ventas de mayor importe, cinco corresponden a Santiago, tres a Concepción y dos a Viña del Mar.

Se observa una influencia importante de los precios unitarios atípicos. Las siete transacciones previamente identificadas mediante IQR aparecen en este ranking.

Por tanto, los resultados deben interpretarse con precaución: los mayores importes pueden reflejar anomalías en los precios registrados y no necesariamente ventas representativas del comportamiento comercial habitual.

### 5.2 Consulta SQL 2: importe vendido y transacciones por sucursal
`COUNT(DISTINCT id_transaccion)` evita contar líneas de una misma transacción como ventas separadas. Se incluye el número de filas como control.


In [0]:
%sql
SELECT
    sucursal,
    SUM(precio_total) AS importe_total,
    COUNT(DISTINCT id_transaccion) AS transacciones,
    COUNT(*) AS filas_venta
FROM ventas_limpias
GROUP BY sucursal
ORDER BY importe_total DESC, sucursal ASC

sucursal,importe_total,transacciones,filas_venta
Santiago,358320486,112,112
Concepción,156110410,70,70
Viña del Mar,124594447,92,92


### Interpretación de ventas por sucursal

El importe total registrado por las tres sucursales alcanza **$639.025.343**, distribuido de la siguiente manera:

| Sucursal | Transacciones | Importe total | Participación |
|---|---:|---:|---:|
| Santiago | 112 | $358.320.486 | 56,07 % |
| Concepción | 70 | $156.110.410 | 24,43 % |
| Viña del Mar | 92 | $124.594.447 | 19,50 % |

Santiago lidera tanto en importe vendido como en cantidad de transacciones. Viña del Mar registra más transacciones que Concepción, pero un importe total inferior.

Esta diferencia evidencia que un mayor volumen de transacciones no implica necesariamente mayores ingresos.

La suma de los importes por sucursal coincide con el total del DataFrame validado, confirmando la coherencia del agregado.

No obstante, la presencia de precios unitarios atípicos puede distorsionar la participación relativa de las sucursales. Por ello, estos resultados representan el comportamiento de los registros procesados y no una medición libre de anomalías.


### 5.3 Consulta SQL 3: estadísticos del precio unitario por categoría
  Se incluyen las ventas imputadas como `Sin categoría`; interpretar ese grupo por separado.


In [0]:

%sql
SELECT
    categoria,
    ROUND(AVG(precio_unitario), 2) AS precio_promedio,
    MIN(precio_unitario) AS precio_minimo,
    MAX(precio_unitario) AS precio_maximo,
    COUNT(*) AS filas,
    SUM(CASE WHEN es_outlier_precio THEN 1 ELSE 0 END) AS precios_atipicos
FROM ventas_limpias
GROUP BY categoria
ORDER BY precio_promedio DESC, categoria ASC



categoria,precio_promedio,precio_minimo,precio_maximo,filas,precios_atipicos
Sin categoría,1148463.95,50,9999990,40,4
Audio,549928.40,1,9999990,50,2
Redes,392705.98,1,9999990,42,1
Almacenamiento,200602.24,25990,449990,49,0
Periféricos,177102.44,50,449990,45,0
Computación,144365.23,1,449990,48,0


### Interpretación de precios unitarios por categoría

La categoría **Sin categoría** presenta el precio unitario promedio más elevado, con $1.148.463,95, seguida de Audio ($549.928,40) y Redes ($392.705,98).

Sin embargo, estos promedios están influenciados por valores atípicos:

- Sin categoría: 4 precios atípicos.
- Audio: 2 precios atípicos.
- Redes: 1 precio atípico.
- Las demás categorías no presentan valores atípicos según los umbrales IQR calculados globalmente.

El precio máximo observado alcanza $9.999.990, considerablemente superior al tercer cuartil de $299.990.

También existen precios mínimos de $1 y $50 en algunas categorías, que merecen revisión aunque no hayan sido identificados como atípicos mediante IQR.

La categoría Sin categoría agrupa registros originalmente incompletos, por lo que su promedio no representa una familia comercial homogénea.

En consecuencia, no sería correcto concluir que estas categorías contienen necesariamente los productos más costosos sin revisar primero la validez de los precios extremos.


### 5.4 Comprobación programática del agregado por sucursal y límites de categorías.

In [0]:
# 5.4 Comprobación programática del agregado
# por sucursal y límites de categorías.

total_df = df_ventas_limpio.agg(
    F.sum("precio_total").alias("total")
).first().total

resultado_sucursales = spark.sql("""
    SELECT sucursal, SUM(precio_total) AS importe_total
    FROM ventas_limpias
    GROUP BY sucursal
""")

total_sucursales = resultado_sucursales.agg(
    F.sum("importe_total").alias("total")
).first().total

assert total_df == total_sucursales, (
    "No coincide el total de sucursales con el total del DataFrame"
)

resultado_categorias = spark.sql("""
    SELECT
        categoria,
        MIN(precio_unitario) AS minimo,
        AVG(precio_unitario) AS promedio,
        MAX(precio_unitario) AS maximo
    FROM ventas_limpias
    GROUP BY categoria
""")

assert resultado_categorias.filter(
    "minimo > promedio OR promedio > maximo"
).count() == 0

print("Conciliación SQL y validación de estadísticas por categoría: OK")


Conciliación SQL y validación de estadísticas por categoría: OK


### 5.5 Impacto económico de los valores atípicos

In [0]:

# Impacto económico de los valores atípicos

resumen_outliers = df_ventas_limpio.agg(
    F.sum("precio_total").alias("venta_total"),
    F.sum(
        F.when(
            F.col("es_outlier_precio"),
            F.col("precio_total")
        ).otherwise(F.lit(0))
    ).alias("venta_atipica")
)

resumen_outliers = resumen_outliers.withColumn(
    "porcentaje_atipico",
    F.round(
        F.col("venta_atipica") /
        F.col("venta_total") * 100,
        2
    )
)

display(resumen_outliers)


venta_total,venta_atipica,porcentaje_atipico
639025343,385555274,60.33


Los 7 registros marcados por IQR representan el **2,55 % de las 274 transacciones** y concentran **$385.555.274**, equivalentes al **60,33 % del importe registrado**. Además, los siete aparecen en el top 10. Esto muestra cuánto influyen los precios extremos en el ranking y en los importes por sucursal. Los registros se conservaron con una marca para su revisión.

## 6. Conclusiones

- **Integración y calidad de los datos.** Se integraron 315 registros de Viña del Mar, Santiago y Concepción. Tras eliminar 15 duplicados completos y excluir 26 filas con cantidades no positivas, quedaron 274 transacciones, sin duplicados ni importes nulos o negativos. Las categorías faltantes se identificaron como `Sin categoría` para conservar las ventas sin asignarlas a una familia de productos sin respaldo.

- **Resultados por sucursal.** Las transacciones registran un importe total de **$639.025.343**. Santiago lidera en importe y número de transacciones. Viña del Mar registra más transacciones que Concepción (92 frente a 70), pero un importe menor ($124.594.447 frente a $156.110.410).

- **Influencia de los precios atípicos.** Las 7 transacciones marcadas por IQR representan el **2,55 %** de las transacciones y concentran el **60,33 %** del importe registrado. Las siete aparecen en el top 10 y se conservaron con una marca para su revisión. Al excluirlas solo para comparar escenarios, Santiago sigue primero y Viña del Mar pasa al segundo lugar. Esto muestra que el orden entre Viña del Mar y Concepción depende de esos registros y que conviene verificar sus precios antes de usar el ranking para evaluar el desempeño comercial.